# Assignment 05 Solutions — Cramer and LU Decomposition

Aligned with `assignments/ASSIGNMENTS.tex`. Replace TODOs with complete solutions.


## Tasks
- Implement cramer_2x2 and cramer_3x3; compare with built-in solver.
- Compute LU decomposition; verify PA=LU and det(A).
- Benchmark Cramer, Gauss, and built-in solver for multiple sizes.
- Solve multiple RHS with one LU factorization.
- Create heatmap of A, L, U.
- Write reflexao.md (200–300 words).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../packages/python/src").resolve()))

import numpy as np
from linalg_utils.systems import cramer_2x2, cramer_3x3

# Cramer's rule — 2x2
A2 = np.array([[1, 2], [3, 4]], dtype=float)
b2 = np.array([5, 6], dtype=float)
x2 = cramer_2x2(A2, b2)
print(f"Cramer 2x2: x = {x2}")
np.testing.assert_allclose(A2 @ x2, b2, atol=1e-9)

# Cramer's rule — 3x3
A3 = np.array([[2, 1, -1], [1, 2, 3], [3, -1, 2]], dtype=float)
b3 = np.array([3, 9, 4], dtype=float)
x3 = cramer_3x3(A3, b3)
print(f"Cramer 3x3: x = {x3}")
np.testing.assert_allclose(A3 @ x3, b3, atol=1e-9)

# Compare with built-in
x_np = np.linalg.solve(A3, b3)
print(f"NumPy solve: x = {x_np}")
np.testing.assert_allclose(x3, x_np, atol=1e-9)
print("Cramer and NumPy solutions match.")

In [ ]:
from linalg_utils.lu import decomposicao_lu, det_via_lu

# LU decomposition
A = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], dtype=float)
P, L, U = decomposicao_lu(A)

print(f"P:\n{P}\n\nL:\n{L}\n\nU:\n{U}")
print(f"\nP @ A:\n{P @ A}")
print(f"L @ U:\n{L @ U}")
np.testing.assert_allclose(P @ A, L @ U, atol=1e-9)
print("\nVerified: P @ A = L @ U")

# Determinant via LU
det_lu = det_via_lu(A)
det_np = np.linalg.det(A)
print(f"\ndet via LU:    {det_lu:.6f}")
print(f"det via NumPy: {det_np:.6f}")
np.testing.assert_allclose(det_lu, det_np, atol=1e-9)

In [ ]:
import time
from linalg_utils.systems import cramer, resolver_gauss

# Benchmark Cramer, Gauss, and NumPy for different sizes
sizes = [3, 5, 10, 20, 50]
results = []

rng = np.random.default_rng(42)
for n in sizes:
    A = rng.normal(size=(n, n))
    b = rng.normal(size=n)

    t0 = time.perf_counter()
    for _ in range(10):
        cramer(A, b)
    t_cramer = (time.perf_counter() - t0) / 10

    t0 = time.perf_counter()
    for _ in range(10):
        resolver_gauss(A, b)
    t_gauss = (time.perf_counter() - t0) / 10

    t0 = time.perf_counter()
    for _ in range(10):
        np.linalg.solve(A, b)
    t_numpy = (time.perf_counter() - t0) / 10

    results.append((n, t_cramer, t_gauss, t_numpy))
    print(f"n={n:3d}: Cramer={t_cramer:.6f}s  Gauss={t_gauss:.6f}s  NumPy={t_numpy:.6f}s")

In [ ]:
from linalg_utils.lu import resolver_lu_multiplos_rhs

# Solve multiple RHS with one LU factorization
A = np.array([[4, 1, 0], [1, 3, 1], [0, 1, 2]], dtype=float)
B = np.array([[1, 2, 0], [0, 1, 3], [1, 0, 1]], dtype=float)

X = resolver_lu_multiplos_rhs(A, B)
print(f"A:\n{A}\n\nB (multiple RHS):\n{B}\n\nX (solutions):\n{X}")

# Verify each column
np.testing.assert_allclose(A @ X, B, atol=1e-9)
print("\nVerified: A @ X = B for all right-hand sides.")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Heatmap of A, L, U
A = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], dtype=float)
P, L, U = decomposicao_lu(A)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, mat, title in zip(axes, [A, L, U], ["A", "L", "U"]):
    im = ax.imshow(mat, cmap="RdBu_r", aspect="equal")
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat[i,j]:.2f}", ha="center", va="center", fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig("lu_heatmap.png", dpi=120, bbox_inches="tight")
plt.close()
print("Saved lu_heatmap.png")

In [ ]:
# Reflexao — see reflexao.md
# Summary: Cramer's rule is elegant for small systems but scales as O(n! * n),
# making it impractical beyond n~10. LU decomposition factorizes once in O(n^3)
# and solves each RHS in O(n^2), which is ideal for multiple right-hand sides.
# The benchmark clearly showed NumPy's LAPACK-backed solver outperforming both
# implementations, demonstrating why optimized libraries matter in practice.